# Structured Explanation contract

This notebook demonstrates the unified **consumption contract**, not a claim that every engine has the same proof semantics. It runs the same Rule and Policy through two sealed paths:

- a V1 Native detached Explain with a captured, sanitized `EvidenceGraph`;
- a Product V2 Native observation whose EvidenceGraph is honestly unavailable.

Business code reads `to_dict()` and availability state. `narrate()` and `render_text()` remain display-only.

In [1]:
from __future__ import annotations

import json
import sys
from hashlib import sha256
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / 'src').exists() and (repo_root.parent / 'src').exists():
    repo_root = repo_root.parent
src_dir = repo_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from factgraph.sdk import (
    AssetMeta,
    Entity,
    EntityRef,
    Field,
    Identity,
    SDKStore,
    native_deterministic_profile_v1,
    outcome_from_run_v2,
    vars,
)
from factgraph.application import (
    evaluation_explanation_data_v2_from_run,
    result_view_v2_from_run,
)

## 1. Author one Rule and one Policy

The Product authoring API stays graph-bound so semantic ports are resolved against the exact schema. Nothing here registers a global Rule or mutates a Policy registry.

In [2]:
class Person(Entity):
    person_id: str = Identity()
    age: int = Field()


fg = SDKStore([Person])
alice_e_ref = fg.entities.create(Person, person_id='alice')
fg.fields.set(Person.age, alice_e_ref, 22)
alice = EntityRef('Person', {'person_id': 'alice'})

with vars('person', 'age') as (person, age):
    person_values = fg.build_rule(
        id='person_values',
        version='1',
        meta=AssetMeta(name='Person values'),
        when=(Person(person), Person(person).age == age),
        ports={'person': person, 'age': age},
        semantic_ports={'person': Person, 'age': Person.age},
    )

draft = fg.policy_builder(
    'adult_people',
    version='1',
    meta=AssetMeta(name='Adult people'),
)
people = draft.use(person_values, as_='people')
adult_people = draft.build(draft.all(people, people.age > 12))

assert fg.fields.get(Person.age, alice_e_ref) == 22
assert adult_people.asset_meta.name == 'Adult people'

## 2. Graph available: sealed V1 Native detached Explain

The caller selects an explicit row and converts only that row anchor into an Explain target. The adapter exposes a sanitized graph rather than generic engine/source objects.

In [3]:
v1_outcome = (
    fg.query(adult_people)
    .bind(people.person, alice)
    .select('age', people.age)
    .plan(profile=native_deterministic_profile_v1())
    .run()
)
v1_result = result_view_v2_from_run(v1_outcome.run, side='effective')
v1_explanation = evaluation_explanation_data_v2_from_run(
    v1_outcome.run,
    target=v1_result.rows[0].to_explain_target(),
)

assert v1_explanation.evidence.state == 'native_detached_recomputed'
assert v1_explanation.evidence.reason_code is None
assert v1_explanation.evidence.graph is not None
assert v1_explanation.evidence.proof_parity == 'not_claimed'

v1_graph = v1_explanation.evidence.graph
{
    'source_protocol': v1_explanation.source_protocol,
    'evidence_state': v1_explanation.evidence.state,
    'graph_id': v1_graph.graph_id,
    'engine': v1_graph.engine,
    'path_count': len(v1_graph.paths),
}

{'source_protocol': 'evaluation_run_v1',
 'evidence_state': 'native_detached_recomputed',
 'graph_id': 'sha256:4bec44bc490016fc77533434d53f65ede77a15aeded73c5a2c6cc2eef829ef30',
 'engine': 'native',
 'path_count': 1}

## 3. Canonical business projection

`to_dict()` contains detached dictionaries, arrays and JSON scalar values. `content_digest` hashes exactly `to_canonical_bytes()`; it identifies this read projection but does not authenticate the run or source.

In [4]:
v1_data = v1_explanation.to_dict()
v1_bytes = v1_explanation.to_canonical_bytes()
assert json.loads(v1_bytes) == v1_data
assert v1_explanation.content_digest == f"sha256:{sha256(v1_bytes).hexdigest()}"
assert v1_data['$schema'] == 'factgraph.product_explanation'
assert v1_data['schema_version'] == 2
assert v1_data['source_protocol'] == 'evaluation_run_v1'
assert v1_data['evidence']['state'] == 'native_detached_recomputed'
assert v1_data['evidence']['graph']['graph_id'] == v1_graph.graph_id

{
    'content_digest': v1_explanation.content_digest,
    'policy_nodes': len(v1_data['policy']['topology']),
    'graph_paths': len(v1_data['evidence']['graph']['paths']),
}

{'content_digest': 'sha256:3621f2a486fc2986396db5499e63002b7081f5a23b04c7ac1e4fae5135a700ec',
 'policy_nodes': 3,
 'graph_paths': 1}

## 4. Graph unavailable: Product V2 Native observation

The Product V2 result has a sealed observation, world and profile, but its detached Native EvidenceGraph path is not implemented. The same consumption envelope retains the reason instead of returning an ambiguous `None` or fabricating a proof.

In [5]:
v2_profile = fg.execution.native_deterministic(
    target=adult_people,
    name='adult-native-v2',
).build()
v2_run = (
    fg.query(adult_people)
    .bind(people.person, alice)
    .select('age', people.age)
    .plan(profile=v2_profile)
    .run()
)
v2_outcome = outcome_from_run_v2(v2_run)
v2_explanation = v2_outcome.explain(v2_outcome.effective.rows[0])
v2_data = v2_explanation.to_dict()

assert v2_data['source_protocol'] == 'evaluation_run_v2'
assert v2_data['evidence']['state'] == 'not_available'
assert v2_data['evidence']['reason_code'] == (
    'NATIVE_V2_DETACHED_EVIDENCE_GRAPH_NOT_IMPLEMENTED'
)
assert v2_data['evidence']['graph'] is None
assert v2_data['evidence']['proof_parity'] == 'not_claimed'

{
    'source_protocol': v2_explanation.source_protocol,
    'evidence': v2_data['evidence'],
    'content_digest': v2_explanation.content_digest,
}

{'source_protocol': 'evaluation_run_v2',
 'evidence': {'state': 'not_available',
  'reason_code': 'NATIVE_V2_DETACHED_EVIDENCE_GRAPH_NOT_IMPLEMENTED',
  'graph': None,
  'proof_parity': 'not_claimed'},
 'content_digest': 'sha256:65e386f6733e9d3e8c76b233ac88ede3f497e6e5481c55dbf3543177973c236d'}

## 5. Presentation branching

Business behavior dispatches by source protocol and branches on structured evidence state. It never parses narration and never treats a missing graph as a negative conclusion.

In [6]:
GRAPH_BEARING_EVIDENCE_STATES = frozenset({
    'native_detached_recomputed',
    'portable_native_inner_not_parity',
    'problog_trace_captured',
})


def evidence_panel(explanation):
    data = explanation.to_dict()
    if data['source_protocol'] == 'evaluation_run_v1':
        protocol_section = data['execution']
    elif data['source_protocol'] == 'evaluation_run_v2':
        protocol_section = data['profile']
    else:
        raise ValueError('unsupported Product Explain source protocol')
    assert isinstance(protocol_section, dict)

    evidence = data['evidence']
    if evidence['state'] in GRAPH_BEARING_EVIDENCE_STATES:
        graph = evidence['graph']
        assert graph is not None
        return {
            'kind': 'graph',
            'state': evidence['state'],
            'graph_id': graph['graph_id'],
        }
    assert evidence['graph'] is None
    return {
        'kind': 'availability',
        'state': evidence['state'],
        'reason_code': evidence['reason_code'],
    }


panels = [evidence_panel(v1_explanation), evidence_panel(v2_explanation)]
assert panels[0]['kind'] == 'graph'
assert panels[1]['kind'] == 'availability'
assert 'Evidence:' in v1_explanation.render_text()
assert isinstance(v2_explanation.narrate(), tuple)

{
    'panels': panels,
    'v1_narration': v1_explanation.narrate(),
    'v2_rendered_text': v2_explanation.render_text(),
}

{'panels': [{'kind': 'graph',
   'state': 'native_detached_recomputed',
   'graph_id': 'sha256:4bec44bc490016fc77533434d53f65ede77a15aeded73c5a2c6cc2eef829ef30'},
  {'kind': 'availability',
   'state': 'not_available',
   'reason_code': 'NATIVE_V2_DETACHED_EVIDENCE_GRAPH_NOT_IMPLEMENTED'}],
 'v1_narration': ('Evaluation sha256:4c2785c576b3f0f8aafa0ee5060c3806dbaf98be82d4a7866c52440ef6edb17e [effective]',
  'Target: policy adult_people (sha256:16da83097737915c6418b9a131890738fe71f82246f2fa2771df500e447a18ac)',
  'Observation: positive_row_observed; conclusion: holds',
  'Result: native_deterministic_v1 / completeness=complete',
  'Evidence: native_detached_recomputed',
  'Policy topology: 3 nodes (c145ba9390e427922f359eee8eef35ca0a6a57b39b184b227cc41581ed24d39a)',
  'Scenario patch: not_captured',
  'Boundary: no negative proof, source authority, or action authorization is claimed'),
 'v2_rendered_text': 'Evaluation sha256:e237a1adb8b90c0aad26554abeed2a1afd789ff50da24c6b2bf550209c4d58ba

## Boundaries retained

- The common shape is a consumption contract, not proof parity.
- `content_digest` identifies the read projection, not source authenticity.
- Summary and zero-row views do not become negative proofs.
- EvidenceGraph remains a logical evidence projection, not a SourceRecord database.
- This notebook does not establish a Meander CompletedRun, FactBinding, SourceRecord, Agent, or MCP integration.